In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# Results — Metabolic History Model

Standalone post-hoc analysis notebook. Loads the saved model artifacts and test set bundle from `artifacts/` and performs False Positive analysis.

**Artifacts required:**
- `*_model.txt` — LightGBM native model
- `*_test_bundle.joblib` — X_test, y_test, y_test_proba, y_pred_final, feature_names
- `*_metadata.joblib` — optimal_threshold, CAT_FEATURES, random_state, etc.

## 1. Setup

In [ ]:
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

## 2. Load Artifacts

In [ ]:
ARTIFACTS_DIR = Path("./artifacts")

# Set to a specific experiment prefix to force selection.
# Example: "lr_vif_bic_metabolic_v2_20260308_142645"

SELECTED_EXPERIMENT = "exp9_lab_target_undiagnosed_v2_20260527_142933"
# SELECTED_EXPERIMENT =  "exp9_lab_target_v2_20260527_150437"
# SELECTED_EXPERIMENT = "exp7_metabolic_history_v3_PA_zeros_20260526_144735"

# SELECTED_EXPERIMENT = "lr_vif_bic_metabolic_PA_adjusted_20260527_143637"
# SELECTED_EXPERIMENT = "lr_vif_bic_lab_target_v2_20260527_192243"
# SELECTED_EXPERIMENT = "lr_vif_bic_undiagnosed_v2_20260527_193532"

DATASET_PATH = Path("../../../dataset/output/processed_data_combined_metabolic_history_v2.csv")

suffix_to_key = {
    "_metadata.joblib": "metadata",
    "_test_bundle.joblib": "test_bundle",
    "_model.txt": "model_txt",
    "_model.joblib": "model_joblib",
    "_scaler.joblib": "scaler",
}

artifact_sets = {}
for path in ARTIFACTS_DIR.glob("*"):
    matched_suffix = next((s for s in suffix_to_key if path.name.endswith(s)), None)
    if matched_suffix is None:
        continue

    prefix = path.name[: -len(matched_suffix)]
    if prefix not in artifact_sets:
        artifact_sets[prefix] = {
            "metadata": None,
            "test_bundle": None,
            "model_txt": None,
            "model_joblib": None,
            "scaler": None,
        }
    artifact_sets[prefix][suffix_to_key[matched_suffix]] = path

assert artifact_sets, f"No recognized artifact files found in {ARTIFACTS_DIR}"

complete_experiments = sorted(
    prefix
    for prefix, files in artifact_sets.items()
    if files["metadata"] and files["test_bundle"] and (files["model_txt"] or files["model_joblib"])
)
assert complete_experiments, "No complete artifact sets found (metadata + test bundle + model)"

selected_experiment = SELECTED_EXPERIMENT or complete_experiments[-1]
assert selected_experiment in complete_experiments, (
    f"Unknown or incomplete experiment: {selected_experiment}. "
    f"Available complete experiments: {complete_experiments}"
)

selected_artifacts = artifact_sets[selected_experiment]
metadata_path = selected_artifacts["metadata"]
test_bundle_path = selected_artifacts["test_bundle"]
model_path = selected_artifacts["model_txt"] or selected_artifacts["model_joblib"]
model_kind = "lightgbm_txt" if selected_artifacts["model_txt"] else "joblib"
scaler_path = selected_artifacts["scaler"]

print("Complete experiments found:")
for exp in complete_experiments:
    marker = "*" if exp == selected_experiment else " "
    print(f"{marker} {exp}")

print(f"\nSelected experiment: {selected_experiment}")
print(f"  metadata:    {metadata_path}")
print(f"  test bundle: {test_bundle_path}")
print(f"  model ({model_kind}): {model_path}")
print(f"  scaler:      {scaler_path}")

In [ ]:
metadata = joblib.load(metadata_path)
test_bundle = joblib.load(test_bundle_path)

if model_kind == "lightgbm_txt":
    model = lgb.Booster(model_file=str(model_path))
else:
    model = joblib.load(model_path)

# Keep a legacy alias for any downstream LightGBM-specific code.
booster = model if model_kind == "lightgbm_txt" else None

# Unpack test bundle
X_test: pd.DataFrame = test_bundle["X_test"]
y_test: pd.Series = test_bundle["y_test"]
y_test_proba: np.ndarray = test_bundle["y_test_proba"]
y_pred_final: np.ndarray = test_bundle["y_pred_final"]

# Resolve categorical feature names from metadata when present.
# For LR/RF pipelines, evaluation bundles are expected to be fully numeric.
cat_feature_keys = (
    "cat_features",
    "categorical_features",
    "categorical_cols",
    "categorical_columns",
    "cat_cols",
)
CAT_FEATURES: list[str] = []
cat_features_source = "metadata_missing"
for key in cat_feature_keys:
    value = metadata.get(key)
    if isinstance(value, (list, tuple)):
        CAT_FEATURES = [str(col) for col in value]
        cat_features_source = f"metadata[{key}]"
        break

if not CAT_FEATURES and model_kind == "lightgbm_txt":
    # LGBM can still use categorical columns if they remain object/category dtype at inference time.
    CAT_FEATURES = [
        col
        for col in X_test.columns
        if isinstance(X_test[col].dtype, pd.CategoricalDtype)
        or pd.api.types.is_object_dtype(X_test[col].dtype)
        or pd.api.types.is_bool_dtype(X_test[col].dtype)
    ]
    cat_features_source = "inferred_from_X_test_dtypes"
elif not CAT_FEATURES:
    # For numeric-only bundles (e.g., LR/RF), keep this empty by design.
    cat_features_source = "numeric_bundle_default_empty"

# Unpack metadata with cross-experiment fallbacks
threshold_source = "metadata[optimal_threshold]" if "optimal_threshold" in metadata else "default_0.5"
OPTIMAL_THRESH: float = float(metadata.get("optimal_threshold", 0.5))
RANDOM_STATE: int = int(metadata.get("random_state", 42))

target_candidates = (
    metadata.get("TARGET_COL"),
    metadata.get("target_col"),
    metadata.get("target_column"),
    metadata.get("target"),
    "has_diabetes_or_prediabetes",
)
TARGET_COL: str = next(str(v) for v in target_candidates if v)
target_col_source = (
    "metadata" if any(metadata.get(k) for k in ("TARGET_COL", "target_column", "target"))
    else "default_has_diabetes_or_prediabetes"
)

print(f"Model loader:        {model_kind}")
print(f"CAT_FEATURES source: {cat_features_source} (n={len(CAT_FEATURES)})")
print(f"TARGET_COL source:   {target_col_source} -> {TARGET_COL}")
print(f"Threshold source:    {threshold_source}")
print(f"Test set:            {X_test.shape}")
print(f"Optimal threshold:   {OPTIMAL_THRESH:.4f}")
print(f"Positive cases:      {y_test.sum()} ({y_test.mean():.1%})")

test_metrics = metadata.get("test_metrics", {})
if test_metrics:
    print(f"\nStored test metrics:")
    for k, v in test_metrics.items():
        print(f"  {k}: {v}")
else:
    print("\nStored test metrics: none found in metadata")

In [ ]:
metadata['TARGET_COL']

## 3. Reload Dataset & Recreate Train Split

Needed to:
- Access `lab_positive` / `undiagnosed` columns for FP enrichment
- Provide `X_train` as a reference distribution for comparison plots

In [ ]:
data = pd.read_csv(DATASET_PATH)
_weights = data.pop("survey_weight")

# Feature engineering (must match training notebook exactly)
data["waist_to_height_ratio"] = data["BMXWAIST"] / data["BMXHT"]
data["age_bmi_interaction"] = data["RIDAGEYR"] * data["BMXBMI"]


def create_age_bins(
    df,
    age_column="RIDAGEYR",
    age_bins=(18, 45, 65, 79),
    age_labels=("young_adult", "middle_age", "senior"),
    elderly_label="elderly",
    unknown_label="age_unknown",
    elderly_top_coded_age=80,
):
    age = df[age_column].copy()
    age_group = pd.Series(index=df.index, dtype="object")
    missing_mask = age.isna()
    elderly_mask = age >= elderly_top_coded_age
    age_group[elderly_mask] = elderly_label
    valid_mask = ~missing_mask & ~elderly_mask
    age_group[valid_mask] = pd.cut(
        age[valid_mask], bins=list(age_bins), labels=age_labels, right=False
    )
    age_group[missing_mask] = unknown_label
    dummies = pd.get_dummies(age_group, prefix="age", dtype=int)
    unknown_col = f"age_{unknown_label}"
    if unknown_col in dummies.columns and missing_mask.sum() == 0:
        dummies = dummies.drop(columns=[unknown_col])
    return pd.concat([df, dummies], axis=1)
    
data = create_age_bins(data)

DROP_FROM_FEATURES = [TARGET_COL, "cycle", "lab_positive", "undiagnosed"]
cols_to_drop = [c for c in DROP_FROM_FEATURES if c in data.columns]

# Drop rows where target is NaN before splitting
data = data[data[TARGET_COL].notna()].copy()



X = data.drop(columns=cols_to_drop)
y = data[TARGET_COL]

# Reproduce train/test split structure used in training notebooks
X_trainval, _X_test, y_trainval, _y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=RANDOM_STATE
)
X_train_full, _X_val_thresh, y_train_full, _y_val_thresh = train_test_split(
    X_trainval, y_trainval, test_size=0.12, stratify=y_trainval, random_state=RANDOM_STATE
)

is_lr_experiment = selected_experiment.startswith("lr_")
if is_lr_experiment:
    # LR reference: train_full (no ES split).
    X_train = X_train_full.copy()
    y_train = y_train_full.copy()
else:
    # RF/LGBM reference: ES training subset.
    X_train, _X_val_es, y_train, _y_val_es = train_test_split(
        X_train_full, y_train_full, test_size=0.125, stratify=y_train_full, random_state=RANDOM_STATE
    )

assert set(_X_test.index) == set(X_test.index), "Test set index mismatch -- check random_state"
print(f"Split strategy: {'LR train_full (no ES)' if is_lr_experiment else 'RF/LGBM with ES split'}")
print(f"Split sizes -- Train ref: {len(X_train)} | Test: {len(X_test)}")
print("Index sanity check passed.")

---
# False Positive Analysis

## 4. Build FP DataFrame

In [ ]:
lab_positive_test = data.loc[X_test.index, "lab_positive"]
undiagnosed_test = data.loc[X_test.index, "undiagnosed"]

FP_df = X_test.copy()
FP_df["target"] = y_test
FP_df["predictions"] = y_pred_final
FP_df["proba"] = y_test_proba

FP_mask = (FP_df["target"] == 0) & (FP_df["predictions"] == 1)
FP_cases = FP_df.loc[FP_mask].copy()
FP_cases["lab_positive"] = lab_positive_test.loc[FP_mask]
FP_cases["undiagnosed"] = undiagnosed_test.loc[FP_mask]

print(f"Total test samples: {len(FP_df)}")
print(f"FP cases:           {FP_mask.sum()} ({FP_mask.mean():.1%} of test set)")
FP_cases.head()

### Lab-positive breakdown among FP cases

FP cases where `lab_positive == 1` are clinically interesting: the model flags them as at-risk despite self-reported no diabetes — and the lab results agree with the model (potentially undiagnosed cases).

In [ ]:
print("lab_positive distribution among FP cases:")
print(FP_cases["lab_positive"].value_counts(dropna=False))

n_lab_pos = int(FP_cases["lab_positive"].sum())
print(f"\n{n_lab_pos} of {len(FP_cases)} FP cases are lab-positive ({n_lab_pos / len(FP_cases):.1%})")
print("\nundiagnosed distribution among FP cases:")
print(FP_cases["undiagnosed"].value_counts(dropna=False))

## 5. Null Revision

In [ ]:
nulls = FP_cases.isna().sum()
nulls[nulls > 0]

## 6. EDA
### Feature split: categorical vs numerical

In [ ]:
ALL_CAT = CAT_FEATURES + [col for col in FP_cases.columns if col.startswith("age_")]
ALL_CAT = [c for c in ALL_CAT if c in FP_cases.columns]

ANALYSIS_COLS = list(X_test.columns)
FP_cat = FP_cases[ALL_CAT]
FP_num = FP_cases[[c for c in ANALYSIS_COLS if c not in ALL_CAT]]

print(f"Numerical features:   {len(FP_num.columns)}")
print(f"Categorical features: {len(FP_cat.columns)}")
FP_num.describe()

### Numerical distributions: FP cases vs Train set

In [ ]:
def plot_side_by_side_density(
    left_series,
    right_series,
    feature_name,
    bins=30,
    integer_bins=False,
    left_label="FP cases",
    right_label="Train set",
):
    left = left_series.dropna()
    right = right_series.dropna()
    if left.empty or right.empty:
        print(f"Skipping {feature_name}: one dataset is empty.")
        return

    if integer_bins:
        min_val = int(min(left.min(), right.min()))
        max_val = int(max(left.max(), right.max()))
        bins = np.arange(min_val, max_val + 2) - 0.5
    else:
        min_val = min(left.min(), right.min())
        max_val = max(left.max(), right.max())
        bins = np.linspace(min_val, max_val, bins)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    axes[0].hist(left, bins=bins, color="tomato", edgecolor="black", alpha=0.8, density=True)
    axes[0].set_title(f"{left_label} — {feature_name} (n={len(left)})")
    axes[0].set_xlabel(feature_name)
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.3)

    axes[1].hist(right, bins=bins, color="steelblue", edgecolor="black", alpha=0.8, density=True)
    axes[1].set_title(f"{right_label} — {feature_name} (n={len(right)})")
    axes[1].set_xlabel(feature_name)
    axes[1].grid(alpha=0.3)

    plt.suptitle(f"{left_label} vs {right_label} — {feature_name}", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_side_by_side_density(
    FP_num["RIDAGEYR"], X_train["RIDAGEYR"],
    feature_name="RIDAGEYR", integer_bins=True,
    left_label="FP set", right_label="Train set",
)
plot_side_by_side_density(
    FP_num["waist_to_height_ratio"], X_train["waist_to_height_ratio"],
    feature_name="waist_to_height_ratio",
    left_label="FP set", right_label="Train set",
)


In [ ]:
def plot_bmx_boxplots(left_df, right_df, left_label="FP cases", right_label="Train set"):
    bmx_features = [
        c for c in ["BMXBMI", "BMXWAIST", "BMXWT", "BMXHT"]
        if c in left_df.columns and c in right_df.columns
    ]
    fig, axes = plt.subplots(1, len(bmx_features), figsize=(5 * len(bmx_features), 5))
    if len(bmx_features) == 1:
        axes = [axes]

    for ax, feature in zip(axes, bmx_features):
        left_vals = left_df[feature].dropna()
        right_vals = right_df[feature].dropna()
        bp = ax.boxplot(
            [left_vals.values, right_vals.values],
            tick_labels=[left_label, right_label],
            patch_artist=True,
            showfliers=False,
            widths=0.6,
        )
        bp["boxes"][0].set_facecolor("tomato")
        bp["boxes"][1].set_facecolor("steelblue")
        for box in bp["boxes"]:
            box.set_alpha(0.8)
        ax.set_title(feature)
        ax.grid(alpha=0.3)

    plt.suptitle(f"BMX Features — {left_label} vs {right_label}", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_bmx_boxplots(FP_num, X_train, left_label="FP set", right_label="Train set")


### Categorical distributions: FP cases vs Train set

In [ ]:
def plot_categorical_distributions(
    left_df, right_df, all_cat, left_label="FP cases", right_label="Train set"
):
    features_to_plot = [
        col
        for col in all_cat
        if col in left_df.columns
        and col in right_df.columns
        and right_df[col].nunique(dropna=True) <= 10
    ]

    n_features = len(features_to_plot)
    if n_features == 0:
        print("No features to plot.")
        return

    fig, axes = plt.subplots(n_features, 2, figsize=(14, 4 * n_features))
    if n_features == 1:
        axes = np.array([axes])

    for i, feature in enumerate(features_to_plot):
        left_vals = left_df[feature].fillna("Missing").astype(str)
        right_vals = right_df[feature].fillna("Missing").astype(str)
        order = sorted(set(left_vals.unique()) | set(right_vals.unique()))

        left_pct = left_vals.value_counts(normalize=True).reindex(order, fill_value=0) * 100
        right_pct = right_vals.value_counts(normalize=True).reindex(order, fill_value=0) * 100

        axes[i, 0].bar(order, left_pct[order], color="tomato", edgecolor="black", alpha=0.8)
        axes[i, 1].bar(order, right_pct[order], color="steelblue", edgecolor="black", alpha=0.8)

        ylim_top = max(left_pct.max(), right_pct.max()) * 1.15
        for ax in [axes[i, 0], axes[i, 1]]:
            ax.set_ylim(0, ylim_top)
            ax.set_ylabel("% of group")
            ax.set_xticks(range(len(order)))
            ax.set_xticklabels(order, rotation=45, ha="right")
            ax.grid(alpha=0.2, axis="y")

        axes[i, 0].set_title(f"{left_label} — {feature} (n={len(left_vals)})")
        axes[i, 1].set_title(f"{right_label} — {feature} (n={len(right_vals)})")

    plt.suptitle(f"Categorical Features — {left_label} vs {right_label}", fontsize=14)
    plt.tight_layout()
    plt.show()


plot_categorical_distributions(FP_cases, X_train, ALL_CAT, left_label="FP set", right_label="Train set")


---
## 7. FP Predicted Probability Distribution

In [ ]:
def plot_proba_distribution(cases_df, threshold, label="FP cases", color="tomato"): # type: ignore
    fig, ax = plt.subplots(figsize=(10, 5)) # type: ignore
    ax.hist(cases_df["proba"], bins=30, color=color, edgecolor="black", alpha=0.8) # type: ignore
    ax.axvline(x=threshold, color="black", linestyle="--", label=f"Threshold = {threshold:.3f}") # type: ignore
    ax.set_xlabel("Predicted Probability", fontsize=12) # type: ignore
    ax.set_ylabel("Count") # type: ignore
    ax.set_title(f"{label} — Predicted Probability Distribution") # type: ignore
    ax.legend() # type: ignore
    ax.grid(alpha=0.3) # type: ignore
    plt.tight_layout() # type: ignore
    plt.show() # type: ignore
    print(f"{label} proba — mean: {cases_df['proba'].mean():.3f} | median: {cases_df['proba'].median():.3f} | max: {cases_df['proba'].max():.3f}") # type: ignore


plot_proba_distribution(FP_cases, OPTIMAL_THRESH, label="FP set")


---

## FP with lab_positive vs no lab_positive

In [ ]:
FP_cases = FP_cases[FP_cases["lab_positive"].notna()].copy()

FP_cases["fp_lab_group"] = np.where(
    FP_cases["lab_positive"].eq(1),
    "FP lab+",
    "FP no lab+",
)

FP_lab_p = FP_cases[FP_cases["fp_lab_group"] == "FP lab+"].copy()
FP_no_lab_p = FP_cases[FP_cases["fp_lab_group"] == "FP no lab+"].copy()

print(FP_cases["fp_lab_group"].value_counts(dropna=False))
print(f"\nFP lab+:    {len(FP_lab_p)} cases")
print(f"FP no lab+: {len(FP_no_lab_p)} cases")


## Visual comparison

In [ ]:
FP_lab_p_num = FP_lab_p[[c for c in ANALYSIS_COLS if c not in ALL_CAT]]
FP_no_lab_p_num = FP_no_lab_p[[c for c in ANALYSIS_COLS if c not in ALL_CAT]]

# Density: FP lab+ vs FP no lab+
plot_side_by_side_density(
    FP_lab_p_num["RIDAGEYR"], FP_no_lab_p_num["RIDAGEYR"],
    feature_name="RIDAGEYR", integer_bins=True,
    left_label="FP lab+", right_label="FP no lab+",
)
plot_side_by_side_density(
    FP_lab_p_num["waist_to_height_ratio"], FP_no_lab_p_num["waist_to_height_ratio"],
    feature_name="waist_to_height_ratio",
    left_label="FP lab+", right_label="FP no lab+",
)

# BMX boxplots: FP lab+ vs FP no lab+
plot_bmx_boxplots(FP_lab_p, FP_no_lab_p, left_label="FP lab+", right_label="FP no lab+")

# Categorical: FP lab+ vs FP no lab+
plot_categorical_distributions(FP_lab_p, FP_no_lab_p, ALL_CAT, left_label="FP lab+", right_label="FP no lab+")

# Proba distributions: FP lab+ vs FP no lab+
plot_proba_distribution(FP_lab_p, OPTIMAL_THRESH, label="FP lab+", color="tomato")
plot_proba_distribution(FP_no_lab_p, OPTIMAL_THRESH, label="FP no lab+", color="steelblue")


## Mann Whitney test

In [ ]:
def mann_whitney_summary(
    left_df,
    right_df,
    num_features,
    left_label="Group A",
    right_label="Group B",
    alpha=0.05,
):
    """Mann-Whitney U per numerical feature. Returns DataFrame sorted by p-value
    with Bonferroni adjustment and significance flag."""
    from scipy.stats import mannwhitneyu
    results = []
    for feat in num_features:
        if feat not in left_df.columns or feat not in right_df.columns:
            continue
        lv = left_df[feat].dropna()
        rv = right_df[feat].dropna()
        if len(lv) < 5 or len(rv) < 5:
            continue
        stat, p = mannwhitneyu(lv, rv, alternative="two-sided")
        results.append({
            "feature": feat,
            f"{left_label}_median": round(lv.median(), 3),
            f"{right_label}_median": round(rv.median(), 3),
            "U_stat": round(stat, 1),
            "p_value": p,
        })
    df = pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)
    df["p_adjusted"] = (df["p_value"] * len(df)).clip(upper=1.0)  # Bonferroni
    df["significant"] = df["p_adjusted"] < alpha
    return df


NUM_FEATURES = [c for c in ANALYSIS_COLS if c not in ALL_CAT]


In [ ]:
print("=== Mann-Whitney U: FP lab+ vs FP no lab+ ===")
mw_fp_labs = mann_whitney_summary(
    FP_lab_p, FP_no_lab_p, NUM_FEATURES,
    left_label="FP lab+", right_label="FP no lab+",
)
display(mw_fp_labs)


In [ ]:
print("=== Mann-Whitney U: FP set vs Train set ===")
mw_fp = mann_whitney_summary(
    FP_cases, X_train, NUM_FEATURES,
    left_label="FP set", right_label="Train set",
)
display(mw_fp)


---

---
# 8. FN vs TP Comparison

Both FN and TP cases are **self-reported diabetics** (`target == 1`).
The question is: what distinguishes the patients the model **missed** (FN) from those it **correctly flagged** (TP)?


In [ ]:
FN_mask = (FP_df["target"] == 1) & (FP_df["predictions"] == 0)
FN_cases = FP_df.loc[FN_mask].copy()
FN_cases["lab_positive"] = lab_positive_test.loc[FN_mask]
FN_cases["undiagnosed"] = undiagnosed_test.loc[FN_mask]

TP_mask = (FP_df["target"] == 1) & (FP_df["predictions"] == 1)
TP_cases = FP_df.loc[TP_mask].copy()
TP_cases["lab_positive"] = lab_positive_test.loc[TP_mask]
TP_cases["undiagnosed"] = undiagnosed_test.loc[TP_mask]

FN_num = FN_cases[[c for c in ANALYSIS_COLS if c not in ALL_CAT]]
TP_num = TP_cases[[c for c in ANALYSIS_COLS if c not in ALL_CAT]]

print(f"FN cases: {FN_mask.sum()} ({FN_mask.mean():.1%} of test set)")
print(f"TP cases: {TP_mask.sum()} ({TP_mask.mean():.1%} of test set)")


In [ ]:
plot_side_by_side_density(
    FN_num["RIDAGEYR"], TP_num["RIDAGEYR"],
    feature_name="RIDAGEYR", integer_bins=True,
    left_label="FN set", right_label="TP set",
)
plot_side_by_side_density(
    FN_num["waist_to_height_ratio"], TP_num["waist_to_height_ratio"],
    feature_name="waist_to_height_ratio",
    left_label="FN set", right_label="TP set",
)


In [ ]:
plot_bmx_boxplots(FN_cases, TP_cases, left_label="FN set", right_label="TP set")


In [ ]:
plot_categorical_distributions(FN_cases, TP_cases, ALL_CAT, left_label="FN set", right_label="TP set")


In [ ]:
plot_side_by_side_density(
    FN_cases["proba"], TP_cases["proba"],
    feature_name="Predicted Probability",
    left_label="FN set", right_label="TP set",
)


In [ ]:
print("=== Mann-Whitney U: FN set vs TP set ===")
mw_fn_tp = mann_whitney_summary(
    FN_cases, TP_cases, NUM_FEATURES,
    left_label="FN set", right_label="TP set",
)
display(mw_fn_tp)


### Lab positive status for FN

In [ ]:
FN_cases['lab_positive'].value_counts()

---
# 7. Four-Group EDA: FP / FN / TP / TN

In [ ]:
# Gate: only run for LightGBM experiments (those with "exp" in name)
RUN_FOUR_GROUP_EDA = "exp" in SELECTED_EXPERIMENT

if not RUN_FOUR_GROUP_EDA:
    print(f"Skipping 4-group EDA: SELECTED_EXPERIMENT='{SELECTED_EXPERIMENT}' does not contain 'exp'")
else:
    print(f"Running 4-group EDA for experiment: {SELECTED_EXPERIMENT}")

In [ ]:
if RUN_FOUR_GROUP_EDA:
    # Top 13 SHAP features (hardcoded from global feature importance plot)
    TOP_SHAP_FEATURES = [
        "age_bmi_interaction",
        "told_high_cholesterol",
        "told_high_bp",
        "RIDAGEYR",
        "BMXWAIST",
        "waist_to_height_ratio",
        "is_female",
        "diastolic_bp",
        "drinking_frequency",
        "vigorous_minutes_per_week",
        "systolic_bp",
        "phq9_score",
        "education_level",
    ]

    SHAP_CATEGORICAL = [
        "told_high_cholesterol",
        "told_high_bp",
        "is_female",
        "drinking_frequency",
        "education_level",
    ]
    SHAP_NUMERICAL = [f for f in TOP_SHAP_FEATURES if f not in SHAP_CATEGORICAL]

    # Rebuild all 4 groups from masks on FP_df (unfiltered test set)
    # Note: FP_cases was reassigned earlier to notna(lab_positive) subset — do not reuse it here.
    _fp = FP_df.loc[FP_mask].copy()
    _fp["lab_positive"] = lab_positive_test.loc[FP_mask]
    _fp["undiagnosed"] = undiagnosed_test.loc[FP_mask]

    _fn = FP_df.loc[FN_mask].copy()
    _fn["lab_positive"] = lab_positive_test.loc[FN_mask]
    _fn["undiagnosed"] = undiagnosed_test.loc[FN_mask]

    _tp = FP_df.loc[TP_mask].copy()
    _tp["lab_positive"] = lab_positive_test.loc[TP_mask]
    _tp["undiagnosed"] = undiagnosed_test.loc[TP_mask]

    TN_mask = (FP_df["target"] == 0) & (FP_df["predictions"] == 0)
    _tn = FP_df.loc[TN_mask].copy()
    _tn["lab_positive"] = lab_positive_test.loc[TN_mask]
    _tn["undiagnosed"] = undiagnosed_test.loc[TN_mask]

    GROUPS = {"TP": _tp, "TN": _tn, "FP": _fp, "FN": _fn}

    print("Group sizes:")
    for name, df in GROUPS.items():
        print(f"  {name}: {len(df):,} ({len(df)/len(FP_df):.1%})")

    print(f"\nNumerical features ({len(SHAP_NUMERICAL)}): {SHAP_NUMERICAL}")
    print(f"Categorical features ({len(SHAP_CATEGORICAL)}): {SHAP_CATEGORICAL}")

    # Sanity check: all groups must sum to test set size
    total = sum(len(df) for df in GROUPS.values())
    assert total == len(FP_df), f"Group sizes sum to {total}, expected {len(FP_df)}"
    print(f"\nSanity check passed: {total} == {len(FP_df)}")

In [ ]:
if RUN_FOUR_GROUP_EDA:
    def compute_summary_stats(groups_dict, features):
        """Compute median and IQR for each feature across groups."""
        rows = []
        for feat in features:
            row = {"feature": feat}
            for group_name, df in groups_dict.items():
                if feat in df.columns:
                    vals = df[feat].dropna()
                    if len(vals) > 0:
                        q1, median, q3 = vals.quantile([    0.25, 0.5, 0.75])
                        row[f"{group_name}_median"] = round(median, 3)
                        row[f"{group_name}_IQR"] = f"[{q1:.2f}, {q3:.2f}]"
                        row[f"{group_name}_n"] = len(vals)
                    else:
                        row[f"{group_name}_median"] = np.nan
                        row[f"{group_name}_IQR"] = "N/A"
                        row[f"{group_name}_n"] = 0
            rows.append(row)
        return pd.DataFrame(rows)

    summary_stats = compute_summary_stats(GROUPS, SHAP_NUMERICAL)
    print("=== Summary Statistics: Numerical SHAP Features ===")
    display(summary_stats)

In [ ]:
if RUN_FOUR_GROUP_EDA:
    def plot_four_group_boxplots(groups_dict, features, ncols=2):
        """Boxplots comparing all 4 groups for each numerical feature."""
        nrows = int(np.ceil(len(features) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
        axes = axes.flatten() if len(features) > 1 else [axes]

        group_order = ["TP", "TN", "FP", "FN"]
        colors = {"TP": "#2ecc71", "TN": "#3498db", "FP": "#e74c3c", "FN": "#f39c12"}

        for idx, feat in enumerate(features):
            ax = axes[idx]
            plot_data = []
            labels = []
            for g in group_order:
                vals = groups_dict[g][feat].dropna()
                plot_data.append(vals)
                labels.append(f"{g}\n(n={len(vals)})")

            bp = ax.boxplot(plot_data, labels=labels, patch_artist=True)
            for patch, g in zip(bp["boxes"], group_order):
                patch.set_facecolor(colors[g])
                patch.set_alpha(0.7)

            ax.set_title(feat, fontsize=11, fontweight="bold")
            ax.set_ylabel("Value")
            ax.grid(axis="y", alpha=0.3)

        for idx in range(len(features), len(axes)):
            axes[idx].set_visible(False)

        plt.suptitle("4-Group Comparison: Numerical SHAP Features", fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

    plot_four_group_boxplots(GROUPS, SHAP_NUMERICAL, ncols=2)

In [ ]:
if RUN_FOUR_GROUP_EDA:
    from itertools import combinations
    from scipy.stats import mannwhitneyu

    def pairwise_mannwhitney(groups_dict, features, alpha=0.05):
        """
        Pairwise Mann-Whitney U tests across all group pairs.
        Returns DataFrame with Bonferroni-corrected p-values.
        """
        group_pairs = list(combinations(groups_dict.keys(), 2))
        n_tests = len(group_pairs) * len(features)

        results = []
        for feat in features:
            for g1, g2 in group_pairs:
                v1 = groups_dict[g1][feat].dropna()
                v2 = groups_dict[g2][feat].dropna()

                if len(v1) < 5 or len(v2) < 5:
                    continue

                stat, p = mannwhitneyu(v1, v2, alternative="two-sided")
                results.append({
                    "feature": feat,
                    "comparison": f"{g1} vs {g2}",
                    f"{g1}_median": round(v1.median(), 3),
                    f"{g2}_median": round(v2.median(), 3),
                    "U_stat": round(stat, 1),
                    "p_value": p,
                })

        df = pd.DataFrame(results)
        if len(df) > 0:
            df["p_adjusted"] = (df["p_value"] * n_tests).clip(upper=1.0)
            df["significant"] = df["p_adjusted"] < alpha
            df = df.sort_values(["feature", "p_value"]).reset_index(drop=True)
        return df

    pairwise_mw = pairwise_mannwhitney(GROUPS, SHAP_NUMERICAL)

    print("=== Pairwise Mann-Whitney U Tests (Bonferroni corrected) ===")
    print(f"Total tests: {len(pairwise_mw)}")
    print(f"Significant after correction: {pairwise_mw['significant'].sum()}")
    display(pairwise_mw[pairwise_mw["significant"]])

In [ ]:
if RUN_FOUR_GROUP_EDA:
    from scipy.stats import chi2_contingency

    def categorical_crosstab_chi2(groups_dict, cat_features, alpha=0.05):
        """
        For each categorical feature:
        - Build crosstab of group × category
        - Run chi-square test
        - Return summary with Bonferroni correction
        """
        n_tests = len(cat_features)
        results = []
        crosstabs = {}

        for feat in cat_features:
            dfs = []
            for group_name, df in groups_dict.items():
                if feat in df.columns:
                    temp = df[[feat]].copy()
                    temp["_group"] = group_name
                    dfs.append(temp)

            if not dfs:
                continue

            combined = pd.concat(dfs, ignore_index=True)
            combined[feat] = combined[feat].fillna("missing")

            ct = pd.crosstab(combined["_group"], combined[feat])
            ct = ct.reindex(["TP", "TN", "FP", "FN"])
            crosstabs[feat] = ct

            try:
                chi2, p, dof, expected = chi2_contingency(ct)
                results.append({
                    "feature": feat,
                    "chi2": round(chi2, 2),
                    "dof": dof,
                    "p_value": p,
                })
            except ValueError as e:
                print(f"Chi2 failed for {feat}: {e}")

        df = pd.DataFrame(results)
        if len(df) > 0:
            df["p_adjusted"] = (df["p_value"] * n_tests).clip(upper=1.0)
            df["significant"] = df["p_adjusted"] < alpha
            df = df.sort_values("p_value").reset_index(drop=True)

        return df, crosstabs

    chi2_results, crosstab_tables = categorical_crosstab_chi2(GROUPS, SHAP_CATEGORICAL)

    print("=== Chi-Square Tests: Categorical SHAP Features (Bonferroni corrected) ===")
    display(chi2_results)

In [ ]:
if RUN_FOUR_GROUP_EDA:
    print("=== Crosstabs (row percentages) ===\n")
    for feat, ct in crosstab_tables.items():
        print(f"--- {feat} ---")
        ct_pct = ct.div(ct.sum(axis=1), axis=0).round(3) * 100
        display(ct_pct)
        print()

In [ ]:
if RUN_FOUR_GROUP_EDA:
    def plot_categorical_stacked_bars(crosstab_tables, ncols=2):
        """Stacked bar charts for categorical features across groups."""
        features = list(crosstab_tables.keys())
        nrows = int(np.ceil(len(features) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
        axes = axes.flatten() if len(features) > 1 else [axes]

        for idx, feat in enumerate(features):
            ax = axes[idx]
            ct = crosstab_tables[feat]
            ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
            ct_pct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", edgecolor="white")
            ax.set_title(feat, fontsize=11, fontweight="bold")
            ax.set_xlabel("Group")
            ax.set_ylabel("Percentage")
            ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

        for idx in range(len(features), len(axes)):
            axes[idx].set_visible(False)

        plt.suptitle("4-Group Comparison: Categorical SHAP Features", fontsize=14, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

    plot_categorical_stacked_bars(crosstab_tables, ncols=2)

---

In [ ]:
total_undiagnosed = y_test[y_test == 0].loc[lab_positive_test == 1].value_counts(dropna=False)

percent = total_undiagnosed / y_test.shape[0] * 100

total_undiagnosed_n = int(total_undiagnosed.sum())
percent = (total_undiagnosed_n / y_test.shape[0]) * 100
print(f"Total undiagnosed (lab_positive=1 & target=0): {total_undiagnosed_n} ({percent:.1f}% of test set)")

In [ ]:
y_test[y_test == 0].value_counts()

percent_undiagnosed_among_negatives = (total_undiagnosed_n / (y_test == 0).sum()) * 100
print(f"Percent undiagnosed among target=0: {percent_undiagnosed_among_negatives:.1f}%")
